# 06 · Continual learning across battery groups (Stages B–C: E4, E6–E8)

**Goal:** measure adaptation and retention when battery groups arrive one at a time (RQ2, RQ3; hypotheses H2, H3 in `docs/RESEARCH_PLAN.md`).

**Protocol.**
* Task = one battery group, split chronologically 70/15/15 (as E1). Nothing is shuffled across time.
* Stream = the six tasks in one order. The identity order plus seeded random permutations; every method sees exactly the same orders and seeds.
* After each training step, every task's test period is evaluated, giving the error matrix `R[i, j]` (test RMSE on task *j* after step *i*).
* Methods: fine-tuning (lower bound), experience replay, EWC, learning without forgetting, all on the same MLP backbone; joint training on all tasks is the upper bound.
* Preprocessing and target scaling are fitted on the first task only (no access to future tasks).

**Metrics** (errors, lower is better; `src/evaluation.py`): final average error, plasticity (mean diagonal), backward transfer and forgetting (positive = forgetting), forward transfer relative to the untrained model (negative = earlier tasks help), training time. 95 % bootstrap CIs over orders × seeds; paired Wilcoxon tests against fine-tuning with Holm correction per metric.

**Hyperparameters** (decision D11): each method's own setting (replay buffer size, EWC λ, LwF α) is selected from a grid fixed in advance, using only the tasks' **validation** periods, on group orders **disjoint** from the evaluation orders. Shared settings (backbone, learning rate, epochs, batch size) are identical for all methods and not tuned. Test periods are used only once, in the final evaluation.

Set `EXPERIMENT_PROFILE=smoke` for a fast code check; smoke results are not for reporting.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

import matplotlib.pyplot as plt

# Fixed colour per battery group (never re-assigned by rank or filter).
GROUP_COLORS = dict(zip(dp.BATTERY_GROUPS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3dd",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#8a8981",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "lines.linewidth": 2,
})

import time

from src import evaluation as ev
from src import experiments as X

PROFILE = os.environ.get("EXPERIMENT_PROFILE", "paper")
if PROFILE == "paper":
    cl_config = X.CLConfig(
        methods=("finetune", "replay", "ewc", "lwf"), n_orders=10, order_seed=0, seeds=(0, 1, 2, 3, 4),
        common_kwargs={"epochs": 200, "patience": 20, "lr": 1e-3, "batch_size": 64},
    )
    tuning_config = X.TuningConfig(n_orders=3, seeds=(0,))  # grids: X.DEFAULT_GRIDS
else:
    cl_config = X.CLConfig(
        methods=("finetune", "replay", "ewc", "lwf"), n_orders=2, order_seed=0, seeds=(0, 1),
        common_kwargs={"epochs": 10, "patience": 3},
    )
    tuning_config = X.TuningConfig(
        grids={"replay": {"buffer_per_task": [50, 200]}, "ewc": {"ewc_lambda": [10.0, 1_000.0]}, "lwf": {"alpha": [0.1, 1.0]}},
        n_orders=1, seeds=(0,),
    )
METHOD_COLORS = {"finetune": "#2a78d6", "replay": "#eb6834", "ewc": "#1baf7a", "lwf": "#eda100"}
METHOD_LABELS = {"finetune": "Fine-tuning", "replay": "Replay", "ewc": "EWC", "lwf": "LwF", "joint": "Joint (upper bound)"}
print("profile:", PROFILE)
print(cl_config)
print(tuning_config)

## 2. Load data

In [ ]:
dp.check_data_files(DATA_ROOT, required=["ml_training_dataset.csv"])
data = dp.read_dataset(DATA_ROOT / "ml_training_dataset.csv")
data[dp.GROUP_COL] = data[dp.GROUP_COL].astype(str)
print("rows:", len(data), "| groups:", dp.check_battery_groups(data[dp.GROUP_COL]))

## 3. Tune method hyperparameters on validation data

For each method and grid value, streams are run over the tuning orders and scored by the **final average validation error** across all tasks after the last step (one number reflecting both adaptation and retention). The lowest value wins; ties go to the earlier grid value. A selection at the edge of its grid means the optimum may lie outside the range: extend the grid and re-run before reporting.

In [ ]:
start = time.perf_counter()
tuning = X.tune_hyperparameters(data, cl_config, tuning_config)
print(f"tuning: {len(tuning['grid_runs'])} streams in {time.perf_counter() - start:.0f} s")
print("tuning orders (disjoint from evaluation orders):")
for o in tuning["orders"]:
    print("  ", " > ".join(o))
print("selected:", tuning["selected"])
edge = {m: [k for k, v in flags.items() if v] for m, flags in tuning["at_grid_edge"].items()}
edge = {m: ks for m, ks in edge.items() if ks}
if edge:
    print("WARNING: selected value at the edge of its grid for", edge, "- extend the grid and re-run before reporting.")
display(tuning["grid_summary"].round(4))

In [ ]:
tuned_methods = [m for m in cl_config.methods if m in tuning["selected"]]
fig, axes = plt.subplots(1, len(tuned_methods), figsize=(3.8 * len(tuned_methods), 3.3), squeeze=False)
for ax, m in zip(axes[0], tuned_methods):
    g = tuning["grid_summary"][tuning["grid_summary"]["method"] == m]
    (param,) = tuning_config.grids[m].keys()
    xs = g[f"param_{param}"].astype(float).to_numpy()
    ax.plot(xs, g["val_final_avg_error"], marker="o", markersize=6, color=METHOD_COLORS[m])
    sel = g[g["selected"]]
    ax.plot(sel[f"param_{param}"].astype(float), sel["val_final_avg_error"], marker="o", markersize=11,
            markerfacecolor="none", markeredgecolor="#1a1a19", markeredgewidth=1.5, linestyle="none")
    ax.set_xscale("log")
    ax.set_xlabel(param)
    ax.set_title(METHOD_LABELS[m], loc="left")
axes[0][0].set_ylabel("Validation final avg error (SOH pp)")
fig.suptitle("Validation-only tuning (circled = selected)")
plt.tight_layout()
plt.show()

cl_config = tuning["tuned_cl_config"]
print("evaluation config with tuned values:", cl_config.method_kwargs)

## 4. Run the evaluation grid

`len(methods) × n_orders × len(seeds)` streams of six tasks each, plus joint training per seed. The paper profile takes on the order of tens of minutes on one CPU core.

In [ ]:
start = time.perf_counter()
cl = X.run_cl_experiment(data, cl_config)
runs, joint = cl["runs"], cl["joint"]
print(f"{len(runs)} streams in {time.perf_counter() - start:.0f} s")
print("orders:")
for i, o in enumerate(cl["orders"]):
    print(f"  {i}: {' > '.join(o)}")

## 5. Summary with confidence intervals

Mean and 95 % bootstrap CI over orders × seeds. Joint training has no sequence, so only its final average error (mean over tasks) is shown.

In [ ]:
metric_cols = list(X.CL_METRIC_COLS)
summary = ev.summarise_runs(runs, metric_cols, ["method"])
joint_per_seed = joint.groupby("seed")["error"].mean()
jm, jl, jh = ev.bootstrap_ci(joint_per_seed.to_numpy())
display(summary.set_index("method").round(3))
print(f"Joint training final average error: {jm:.3f} [{jl:.3f}, {jh:.3f}] (n={len(joint_per_seed)} seeds)")

## 6. Adaptation–retention trade-off

Each method's plasticity (error on the task just learned; lower is better adaptation) against forgetting (lower is better retention), with 95 % CIs. The ideal is the lower-left corner.

In [ ]:
s = summary.set_index("method")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
for m in cl_config.methods:
    ax.errorbar(s.loc[m, "plasticity_mean"], s.loc[m, "forgetting_mean"],
                xerr=[[s.loc[m, "plasticity_mean"] - s.loc[m, "plasticity_ci_low"]], [s.loc[m, "plasticity_ci_high"] - s.loc[m, "plasticity_mean"]]],
                yerr=[[s.loc[m, "forgetting_mean"] - s.loc[m, "forgetting_ci_low"]], [s.loc[m, "forgetting_ci_high"] - s.loc[m, "forgetting_mean"]]],
                fmt="o", markersize=8, color=METHOD_COLORS[m], ecolor=METHOD_COLORS[m], elinewidth=1.5, capsize=3, label=METHOD_LABELS[m])
ax.axhline(0, color="#8a8981", linewidth=1)
ax.set_xlabel("Plasticity: error on newest task (SOH pp)")
ax.set_ylabel("Forgetting (SOH pp)")
ax.set_title("Adaptation vs. retention", loc="left")
ax.legend(frameon=False, fontsize=9, loc="best")

ax = axes[1]
xs = np.arange(len(cl_config.methods))
means = [s.loc[m, "final_avg_error_mean"] for m in cl_config.methods]
lo = [s.loc[m, "final_avg_error_mean"] - s.loc[m, "final_avg_error_ci_low"] for m in cl_config.methods]
hi = [s.loc[m, "final_avg_error_ci_high"] - s.loc[m, "final_avg_error_mean"] for m in cl_config.methods]
ax.bar(xs, means, color=[METHOD_COLORS[m] for m in cl_config.methods], width=0.6)
ax.errorbar(xs, means, yerr=[lo, hi], fmt="none", ecolor="#3d3d3a", capsize=3, linewidth=1)
ax.axhline(jm, color="#5f5e5a", linestyle="--", linewidth=1.5)
ax.text(-0.45, jm, "joint training", va="bottom", ha="left", fontsize=9, color="#5f5e5a",
        bbox={"facecolor": "white", "edgecolor": "none", "pad": 1})
ax.set_xticks(xs, [METHOD_LABELS[m] for m in cl_config.methods])
ax.set_ylabel("Final average error (SOH pp)")
ax.set_title("Error on all groups after the last task", loc="left")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

## 7. Error matrices for one stream

`R[i, j]` for the identity order and the first seed. Rows are training steps, columns are tasks; the diagonal is the error on the task just learned, entries below it show retention, entries above it show performance before a task is trained on. One shared colour scale across methods.

In [ ]:
seed0 = cl_config.seeds[0]
mats = {m: cl["matrices"][(m, 0, seed0)] for m in cl_config.methods}
vmax = max(float(np.nanmax(R.to_numpy())) for R in mats.values())
fig, axes = plt.subplots(1, len(mats), figsize=(3.9 * len(mats), 4.4), squeeze=False)
for k, (ax, (m, R)) in enumerate(zip(axes[0], mats.items())):
    im = ax.imshow(R.to_numpy(), cmap="Blues", vmin=0, vmax=vmax)
    ax.set_xticks(range(R.shape[1]), R.columns, fontsize=8)
    ax.set_yticks(range(R.shape[0]), [f"after {t}" for t in R.index] if k == 0 else [], fontsize=8)
    ax.set_xlabel("evaluated task", fontsize=9)
    ax.set_title(METHOD_LABELS[m], loc="left")
    ax.grid(False)
    for (i, j), v in np.ndenumerate(R.to_numpy()):
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7, color="white" if v > 0.6 * vmax else "#1a1a19")
fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8, label="Test RMSE (SOH pp)")
plt.show()

## 8. Paired comparisons against fine-tuning

Differences are method − fine-tuning on matched (order, seed) runs: negative means lower error or less forgetting than fine-tuning. The Holm family is all methods-vs-fine-tuning comparisons for one metric. Report `n_nonzero` with the p-value.

In [ ]:
comparisons = X.compare_methods(runs, baseline="finetune")
display(comparisons[["metric", "method", "n", "n_nonzero", "mean_diff", "ci_low", "ci_high", "p_value", "p_holm", "rank_biserial"]].round(4))

## 9. Hypotheses

* **H2** — fine-tuning forgets: forgetting CI for fine-tuning lies above 0.
* **H3** — a method reduces forgetting relative to fine-tuning (Holm-adjusted p < 0.05, negative difference) without raising plasticity error by more than 0.25 pp.

In [ ]:
ft = s.loc["finetune"]
h2 = bool(ft["forgetting_ci_low"] > 0)
print(f"H2 fine-tuning forgetting {ft['forgetting_mean']:.3f} [{ft['forgetting_ci_low']:.3f}, {ft['forgetting_ci_high']:.3f}] -> supported: {h2}")
margin = 0.25
h3_rows = []
for m in [x for x in cl_config.methods if x != "finetune"]:
    f = comparisons[(comparisons["metric"] == "forgetting") & (comparisons["method"] == m)].iloc[0]
    p = comparisons[(comparisons["metric"] == "plasticity") & (comparisons["method"] == m)].iloc[0]
    h3_rows.append({"method": m, "forgetting_diff": f["mean_diff"], "forgetting_p_holm": f["p_holm"],
                    "plasticity_diff": p["mean_diff"],
                    "supports_H3": bool(f["mean_diff"] < 0 and f["p_holm"] < 0.05 and p["mean_diff"] <= margin)})
h3 = pd.DataFrame(h3_rows)
h3.round(4)

## 10. Save results

In [ ]:
matrices_out = [{"method": m, "order_id": o, "seed": sd, "R": R.round(6).to_dict(orient="split")}
                for (m, o, sd), R in cl["matrices"].items()]
out = X.write_results(RESULTS_DIR / "cl_metrics.json", {
    "profile": PROFILE, "config": cl["config"], "orders": cl["orders"], "runs": runs, "summary": summary,
    "tuning": {k: tuning[k] for k in ("config", "orders", "selected", "at_grid_edge", "grid_summary", "grid_runs")},
    "joint": joint, "comparisons": comparisons, "hypotheses": {"H2": h2, "H3": h3}, "matrices": matrices_out,
}, REPO_ROOT)
print("wrote", out.relative_to(REPO_ROOT), f"({out.stat().st_size / 1e3:.0f} kB)")
if PROFILE != "paper":
    print("NOTE: smoke profile — these numbers are for code checks only.")

## 11. Caveats

* **Six tasks.** Orders × seeds give variability estimates, but all runs share the same six groups; the CIs do not cover other battery populations.
* **Tuning scope.** Only each method's own hyperparameter is tuned, on validation periods of orders disjoint from the evaluation orders; shared settings are fixed. Rankings may still depend on the grids and on the shared settings; the full tuning grid is saved with the results so reviewers can see the sensitivity.
* **Forward-transfer reference** is the untrained backbone with first-task preprocessing, whose predictions stay near the first task's mean SOH.
* **Step-function labels** (D4) affect every task's test period in the same way as in notebook 04.